In [79]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score, recall_score,f1_score,confusion_matrix,classification_report

In [81]:
df=pd.read_csv('loan_approval_dataset.csv')
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [83]:
df.shape

(4269, 13)

In [85]:
col_names=df.columns
col_names

Index(['loan_id', ' no_of_dependents', ' education', ' self_employed',
       ' income_annum', ' loan_amount', ' loan_term', ' cibil_score',
       ' residential_assets_value', ' commercial_assets_value',
       ' luxury_assets_value', ' bank_asset_value', ' loan_status'],
      dtype='object')

In [87]:
num_cols=df.select_dtypes(include=np.number).columns
num_cols

Index(['loan_id', ' no_of_dependents', ' income_annum', ' loan_amount',
       ' loan_term', ' cibil_score', ' residential_assets_value',
       ' commercial_assets_value', ' luxury_assets_value',
       ' bank_asset_value'],
      dtype='object')

In [89]:
cat_cols=df.select_dtypes(include=object).columns
cat_cols

Index([' education', ' self_employed', ' loan_status'], dtype='object')

In [91]:
df[cat_cols].isnull().sum()

education        0
self_employed    0
loan_status      0
dtype: int64

In [93]:
df[num_cols].isnull().sum()

loan_id                      0
 no_of_dependents            0
 income_annum                0
 loan_amount                 0
 loan_term                   0
 cibil_score                 0
 residential_assets_value    0
 commercial_assets_value     0
 luxury_assets_value         0
 bank_asset_value            0
dtype: int64

In [95]:
for col in num_cols:
    Q1=df[col].quantile(0.25)
    Q3=df[col].quantile(0.75)
    IQR=Q3-Q1
    lb=Q1-1.5*IQR
    ub=Q3+1.5*IQR
    outliers = df[(df[col] < lb) | (df[col] > ub)]
    print(col,':',len(outliers))

loan_id : 0
 no_of_dependents : 0
 income_annum : 0
 loan_amount : 0
 loan_term : 0
 cibil_score : 0
 residential_assets_value : 52
 commercial_assets_value : 37
 luxury_assets_value : 0
 bank_asset_value : 5


In [97]:
for col in num_cols:
    Q1=df[col].quantile(0.25)
    Q3=df[col].quantile(0.75)
    IQR=Q3-Q1
    lb=Q1-1.5*IQR
    ub=Q3+1.5*IQR
    df[col]=np.where(df[col]<lb,lb,df[col])
    df[col]=np.where(df[col]>ub,ub,df[col])
    outliers = df[(df[col] < lb) | (df[col] > ub)]
    print(col,':',len(outliers))

loan_id : 0
 no_of_dependents : 0
 income_annum : 0
 loan_amount : 0
 loan_term : 0
 cibil_score : 0
 residential_assets_value : 0
 commercial_assets_value : 0
 luxury_assets_value : 0
 bank_asset_value : 0


In [99]:
scaler=StandardScaler()
df[num_cols]=scaler.fit_transform(df[num_cols])
df[num_cols]

,loan_id,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
0,-1.731645,-0.294102,1.617979,1.633052,0.192617,1.032792,-0.783495,2.770319,0.832028,0.930707
1,-1.730834,-1.473548,-0.341750,-0.324414,-0.508091,-1.061051,-0.736995,-0.633638,-0.694993,-0.515991
2,-1.730022,0.295621,1.439822,1.610933,1.594031,-0.544840,-0.055003,-0.106426,1.996520,2.408185
3,-1.729211,0.295621,1.119139,1.721525,-0.508091,-0.771045,1.665478,-0.381493,0.897943,0.899926
4,-1.728399,1.475067,1.689242,1.002681,1.594031,-1.264055,0.766488,0.741698,1.568075,0.007282
...,...,...,...,...,...,...,...,...,...,...
4264,1.728399,1.475067,-1.446324,-1.419268,0.192617,-1.641063,-0.721495,-1.023316,-1.299210,-1.285511
4265,1.729211,-1.473548,-0.626801,-0.423946,1.594031,-0.237434,-0.504498,-0.473182,-0.453306,-0.946923
4266,1.730022,-0.294102,0.513405,0.969504,1.243677,-0.829046,-0.969492,1.704434,0.326683,0.715241
4267,1.730834,-0.883825,-0.341750,-0.258059,-0.508091,1.044393,0.115495,-0.977472,-0.112748,0.253529


In [101]:
print(df.columns.tolist())

['loan_id', ' no_of_dependents', ' education', ' self_employed', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value', ' loan_status']


In [103]:
print(cat_cols)

Index([' education', ' self_employed', ' loan_status'], dtype='object')


In [105]:
df.columns = df.columns.str.strip()
cat_cols=df.select_dtypes(include=object).columns
for col in cat_cols:
    df[col] = df[col].str.strip()

In [107]:
encoder=LabelEncoder()
for col in cat_cols:
    df[col]=encoder.fit_transform(df[col])

In [109]:
df

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,-1.731645,-0.294102,0,0,1.617979,1.633052,0.192617,1.032792,-0.783495,2.770319,0.832028,0.930707,0
1,-1.730834,-1.473548,1,1,-0.341750,-0.324414,-0.508091,-1.061051,-0.736995,-0.633638,-0.694993,-0.515991,1
2,-1.730022,0.295621,0,0,1.439822,1.610933,1.594031,-0.544840,-0.055003,-0.106426,1.996520,2.408185,1
3,-1.729211,0.295621,0,0,1.119139,1.721525,-0.508091,-0.771045,1.665478,-0.381493,0.897943,0.899926,1
4,-1.728399,1.475067,1,1,1.689242,1.002681,1.594031,-1.264055,0.766488,0.741698,1.568075,0.007282,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4264,1.728399,1.475067,0,1,-1.446324,-1.419268,0.192617,-1.641063,-0.721495,-1.023316,-1.299210,-1.285511,1
4265,1.729211,-1.473548,1,1,-0.626801,-0.423946,1.594031,-0.237434,-0.504498,-0.473182,-0.453306,-0.946923,0
4266,1.730022,-0.294102,1,0,0.513405,0.969504,1.243677,-0.829046,-0.969492,1.704434,0.326683,0.715241,1
4267,1.730834,-0.883825,1,0,-0.341750,-0.258059,-0.508091,1.044393,0.115495,-0.977472,-0.112748,0.253529,0


In [111]:
X=df.drop('loan_status',axis=1)
y=df['loan_status']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
model=LogisticRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

In [113]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9063231850117096
Precision: 0.8838709677419355
Recall: 0.8616352201257862
F1 Score: 0.8726114649681529
